# Théorie :

**Problème d'un MLP sur une image** : il faut l'applatir en vecteur, on perds la structure spatiale et necessite bcp de poids + instable aux transformation géométrique (translation, symétrie ...).  
On a donc partir sur une structure différentes : **Les CNNs**  
On va utiliser un ensemble de filtres (somme pondérées sur l'imagette ...). Le poids de chaque filtre est commun à toute l'image, chaque filtre devient un détecteur de motif (ligne droite, courbe droite ...)  

Un CNN contient plusieurs couches :  
1. **Convolution** : Produit une feature map par filtre. (Notions à maitriser : noyau, stride, padding, nombre de canaux d'entrée et de sortie.)
2. **Activation (ReLu)** : comme dans le MLP
3. **Pooling** (max pooling) : réduit la taille de la carte et donne une tolérance aux petits décalages.
4. **Aplatissement puis couche dense** : à la fin, on retombe sur un MLP qui classe à partir des motifs détectés (Cette fois, on ne fera pas le MLP entier mais simplifié avec la backprof intégrée à PyTorch)
5. **Sortie** : softmax avec 2 classes (carré/cercle), ou 1 neurone sigmoïde. Loss = cross-entropy

**Apprentissage** : même principe que MLP (boucle {forward, loss, backprog, maj des w} jusqu'à un stop). La nouveauté est que le gradient d'une convolution est lui-même une convolution (avec le filtre retourné), et que celui du max pooling n'est transmis qu'à la position du maximum.

## Plan large proposé par claude
1. Fait (comprendre convolution)
1. Fait (jeu donnée carré/cercle)
2. **Pipeline de données** : normalisation, séparation train/validation/test, mélange, batchs.
3. **Modèle minimal** : 1 à 2 blocs conv + ReLU + pooling, puis une couche dense. Commence petit.
4. **Boucle d'entraînement** : perte, optimiseur, suivi des courbes loss/accuracy (train vs validation).
5. **Évaluation** : accuracy, matrice de confusion, et visualisation des images mal classées.
6. **Comprendre ce que le modèle apprend** : afficher les filtres et les feature maps.
7. **Test final sur tes dessins à la main** : prétraitement identique (redimensionnement, inversion des couleurs si besoin), puis prédiction.
8. **Améliorations** (optionnel) : augmentation de données (translation, rotation, bruit, épaisseur du trait), dropout, comparaison avec ton MLP.

![alt text](TP-CNN-architecture-1.jpg)

### *1. Notions liés à la convolution*
#### *Vocabulaire*
**Noyau :**  
- imagette issus de la matrice qu'on fait glisser sur l'image (3x3 par exemple).
- A chaque position légale, on multiplie terme à terme puis on somme, et on rend 1 résultat. Les poids sont les w issus de l'entrainement.

**Stride (pas) :**  
- Combien de case le noyau se déplace

**Padding :**  
- Présence ou non de bordure rajouté autour de l'entrée
  - padding = 'valid' : aucun rajout. Sortie de taille -2 sur chaque axe
  - padding = 'same' : on rajoute des valeurs sur les contours de l'entrée, qui fait que la sorti est de taille identique à l'entrée (np.pad sur l'entrée)

**Canaux :**  
- Nombre c de matrices (x,y) utilisés pour representer l'image de taille (c,x,y)

#### *Taille impliqué dans la convolution :*
1. **Entrée**
- `C_in` : nombre de canaux d'entrée
- `H` (ou `n`) : hauteur de l'image (donc nbr de lignes)
- `W` (ou `self.self.P`) : largeur ... (nbr de colonnes)
  - Forme de l'entrée : `(C_in, H, W)`   
2. **Hyperparamètres de la couche**
- `C_out` : nombre de filtres, donc nbr canaux de sortie
- `k` : taille du noyau (k*k, souvent 3)
- `s` : stride, le pas de déplacement
- `P`: padding, nombre de cases ajoutées sur les bords (avec `'valid'`, P = 0)  
3. **Intermédiaire : ce que la couche contient**
- Noyau : matrice `k*k` pour un canal
- Filtre : `C_in` noyaux, de forme `(C_in, k, k)`
- Tous les filtres ensemble : forme (`(C_out, C_in, k, k)`)
- Biais : `C_out` valeurs, une par filtre
- Nombre de paramètres appris : `C_out * C_in * k * k + C_out`  
4. **Sortie**
- De forme `(C_out, H_out, W_out)`
- $H_{out} = floor[(H - k + 2P)/s] +1$
- $W_{out} = floor[(W - k + 2P)/s] +1$
  - avec `'valid'` et un stride de 1, on retrouve `H-k+1` et `W-k+1`, soit `H-2` et `W-2` pour k = 3

#### *Resumé :*
**Entrée** **$\to$** Si l'image à plusieurs canaux, on repère l'axe des canaux (le plus petit) et on remet dans l'ordre `(C_in, H, W)` au besoin.  

**Noyaux** **$\to$** On crée `C_out` filtres de forme `(C_in, k, k)` : `(C_out, C_in, k, k)` pour l'ensemble ou un dict ...    
**$\to$** Les poids sont initialisés aléatoirement, et rajoute d'un biais par filtre (soit `C_out` biais tous initialisé à 0)  

**Padding** **$\to$** `'valid'` donne `P=0`, `'same'` donne `P = (k-1)//2`, et un entier est utilisé tel quel. On pad uniquement sur `H` et `W` (pas les canaux)  

**Convolution** **$\to$** pour chaque filtre, on fait glisser la fenetre sur `df`. La map de sortie est de taille `(H_out, W_out)`.  
**$\to$** On itère pour chaque fenetre valide, et on recupere la somme pondérée par poids du noyau (multiplication terme a terme noyau * fenetre), et on rajoute le biais.  

**Sortie** **$\to$** On fait un `np.stack` des maps, ce qui donne un matrice 3D `(C_out, H_out, W_out)`. Chaque couche de sortie correspond à un filtre.  


In [ ]:
def convolution(self, M):
    df = M.copy()
    format = list(df.shape[-2:])

    if self.P > 0:
        df = np.pad(df, [(0, 0)] * (df.ndim - 2) + [(self.P, self.P), (self.P, self.P)])

    N_out = []
    for c, noyau in self.noyau.items():
        N = np.zeros((
            (format[0] - self.k + 2 * self.P) // self.stride + 1,
            (format[1] - self.k + 2 * self.P) // self.stride + 1
        ))
            
        for i in range(N.shape[0]):
            for j in range(N.shape[1]):
                fenetre = df[
                    ...,
                    i * self.stride:i * self.stride + self.k,
                    j * self.stride:j * self.stride + self.k
                ]

                # if fenetre.shape[-2:] == (self.k, self.k):
                N[i,j] = np.sum(fenetre * noyau) + self.biais[c]
        N_out.append(N)
    return np.stack(N_out)

### *2. Notion liées au maxpooling*

In [ ]:
def maxpooling(self, M):
    df = M.copy()
    format = list(df.shape)
    format[-1] = (format[-1] - self.k_maxpooling) // self.stride_maxpooling + 1
    format[-2] = (format[-2] - self.k_maxpooling) // self.stride_maxpooling + 1

    N_out = np.zeros(format)
    for i in range(N_out.shape[-2]):
        for j in range(N_out.shape[-1]):
            fenetre = df[
                ...,
                i * self.stride_maxpooling:i * self.stride_maxpooling + self.k_maxpooling,
                j * self.stride_maxpooling:j * self.stride_maxpooling + self.k_maxpooling
            ]                
            N_out[...,i,j] = np.max(fenetre, axis=(-1,-2))
    return N_out

In [2]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_openml
from torch.utils.data import DataLoader, TensorDataset


## Modèle

In [ ]:
class CNN_numpy:
    def __init__(self, C_out=6, kernel=3, stride=1, padding="valid"):
        '''
        C_out : nombre de filtre et donc nombre de canaux en sortie de convolution
        k : taille du filtre / noyau
        '''
        self.M = []
        self.C_out = C_out
        self.stride = stride
        self.k = kernel
        self.padding = padding

        self.k_maxpooling = 2
        self.stride_maxpooling = 2
        self.fit_init = False

        self.pad_dict = {'valid': 0, 'same': (self.k - 1)//2}
        if self.padding in self.pad_dict:
            self.P = self.pad_dict[self.padding]
        elif type(self.padding) == int:
            self.P = self.padding
        else:
            raise KeyError("Definir padding pour 'valid', 'same' ou entier")

    def convolution(self, M):
        df = M.copy()
        format = list(df.shape[-2:])

        if self.P > 0:
            df = np.pad(df, [(0, 0)] * (df.ndim - 2) + [(self.P, self.P), (self.P, self.P)])

        N_out = []
        for c, noyau in self.noyau.items():
            N = np.zeros((
                (format[0] - self.k + 2 * self.P) // self.stride + 1,
                (format[1] - self.k + 2 * self.P) // self.stride + 1
            ))

            for i in range(N.shape[0]):
                for j in range(N.shape[1]):
                    fenetre = df[
                        ...,
                        i * self.stride:i * self.stride + self.k,
                        j * self.stride:j * self.stride + self.k
                    ]

                    # if fenetre.shape[-2:] == (self.k, self.k):
                    N[i,j] = np.sum(fenetre * noyau) + self.biais[c]
            N_out.append(N)
        return np.stack(N_out)

    def maxpooling(self, M):
        df = M.copy()
        format = list(df.shape)
        format[-1] = (format[-1] - self.k_maxpooling) // self.stride_maxpooling + 1
        format[-2] = (format[-2] - self.k_maxpooling) // self.stride_maxpooling + 1

        N_out = np.zeros(format)
        for i in range(N_out.shape[-2]):
            for j in range(N_out.shape[-1]):
                fenetre = df[
                    ...,
                    i * self.stride_maxpooling:i * self.stride_maxpooling + self.k_maxpooling,
                    j * self.stride_maxpooling:j * self.stride_maxpooling + self.k_maxpooling
                ]                
                N_out[...,i,j] = np.max(fenetre, axis=(-1,-2))
        return N_out

    def fit_initialisation(self, M):
        df = M.copy()
        if df.ndim == 3:
            idx_canal = np.argmin(df.shape)
            if idx_canal == 2:                  # Si on est en convention plt.imread ou PIL (H, W, C)
                df = np.moveaxis(df, -1, 0)     # Sinon on a bien (Canaux, Hauteur, Largeur)
   
        if df.ndim == 2:
            self.noyau = {j: [np.random.rand(self.k) for _ in range(self.k)] for j in range(self.C_out)}
        else:
            self.noyau = {j: np.array([[np.random.rand(df.shape[0]) for _ in range(self.k)] for _ in range(self.k)]) for j in range(self.C_out)}
        self.biais = np.zeros(self.C_out)
        return df, True

    def fit(self, M):
        df = M.copy()
        if not self.fit_init:
            df, self.fit_init = self.fit_initialisation(df)
        
        N_out = np.maximum(0,self.convolution(df))  # Activation avec ReLu(x) = max(0,x) sur la sortie de convolution
        
        N_out = self.maxpooling(N_out)              # Maxpooling sur la sortie de ReLu.Convolution

        return N_out


In [167]:
X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False)
X_mnist = X_mnist.reshape(-1, 28, 28) / 255.        # (N, 784) -> (N, 28, 28), valeurs dans [0, 1]
y_mnist = y_mnist.astype(np.int64)

Information pour claude à lire :
On ne veux pas passer par des blocs "boite noir" de pytorch, mais faire a la main les étapes.

In [ ]:
class Conv:
    def __init__(self, C_in=1, C_out=6, kernel=3, stride=1, padding="valid"):
        '''
        C_in : nbr de canaux d'entrée - 3 pour RGB ...
        C_out : nombre de filtre et donc nombre de canaux en sortie de convolution
        kernel : taille k du noyau (k x k)
        stride : pas de déplacement de la fenêtre d'application du noyau
        padding : 'valid' (P=0), 'same' (P=(k-1)//2) ou entier >= 0
        '''
        self.C_in, self.C_out = C_in, C_out
        self.k, self.stride = kernel, stride
        self.padding = padding

        # Traduction du padding str -> int
        pad_dict = {'valid': 0, 'same': (self.k - 1)//2}
        if padding in pad_dict:
            self.P = pad_dict[padding]
        elif isinstance(padding, int):
            self.P = padding
        else:
            raise KeyError("Definir padding pour 'valid', 'same' ou entier")

        # Initialisation des poids (C_out, C_in, k, k)
        self.noyau = (torch.randn(C_out, C_in, kernel, kernel) / (C_in * kernel**2) ** 0.5).requires_grad_()
        self.biais = torch.zers(C_out, requires_grad = True)

    def parameters(self):
        return [self.noyau, self.biais]

    def forward(self, x):                   # x : (N, C_in, H, W)
        if x.shape[1] != self.C_in:
            raise ValueError(f"Conv attend {self.C_in} canaux en entrée, recu {x.shape[1]}")
        if self.P > 0:
            x = torch.nn.functional.pad(x, (self.P, self.P, self.P, self.P))
        H_out = (x.shape[-2] - self.k) // self.stride + 1
        W_out = (x.shape[-1] - self.k) // self.stride + 1 
        if H_out <= 0 or W_out <= 0:
            raise ValueError(f"Image trop petite ({x.shape[-2]}x{x.shape[-1]}) pour un noyau {self.k}x{self.k}")

        rows = []
        for i in range(H_out):
            cols = []
            for j in range(W_out):
                fenetre = x[:, :,                               # (N, C_in, k, k)
                            i*self.stride:i*self.stride+self.k,
                            j*self.stride:j*self.stride+self.k]
                cols.append((fenetre[:, None] * self.noyau).sum(dim=(-3,-2,-1)) + self.biais) # (N, C)
            rows.append(torch.stack(cols, dim=-1))  # (N, C, W_out)
        return torch.stack(rows, dim=-2)            # (N, C, H_out, W_out)

class ReLu:
    def parameters(self):
        return []

    def forward(self, x):
        return torch.relu(x) # max(0,x)

class MaxPool:
    def __init__(self, kernel=2, stride=2):
        self.k = kernel
        self.stride = stride

    def parameters(self):
        return []

    def forward(self, x):   # x : (N, C, H, W), pas de padding possible
        H_out = (x.shap[-2] - self.k) // self.stride + 1
        W_out = (x.shape[-1] - self.k) // self.stride + 1
        if H_out <= 0 or W_out <= 0:
            raise ValueError(f"Image trop petite ({x.shape[-2]}x{x.shape[-1]}) pour un noyau {self.k}x{self.k}")

        rows = []
        for i in range(H_out):
            cols = []
            for j in range(W_out):
                fenetre = x[...,                                # (N, C, k, k)
                            i*self.stride:i*self.stride+self.k,
                            j*self.stride:j*self.stride+self.k]
                cols.append(torch.amax(fenetre, dim=(-2,-1)))   # (N, C)
            rows.append(torch.stack(cols, dims=-1))             # (N, C, W_out)
        return torch.stack(rows,dim=-2)                         # (N, C, H_out, W_out)

class Flatten:
    def parameters(self):
        return []

    def forward(self, x):
        return x.flatten(1)     # (N, C, H, W) -> (N, C*H*W)

class Dense:
    def __init__(self, n_out):
        self.n_in = None    # Connu seulement après le 1er passage
        self.n_out = n_out
        self.W = None
        self.b = None

    def parameters(self):
        if self.W is None:
            raise RuntimeError("Dense enconre non initialisé : faire tourner le modèle (premier passage)")
        return [self.W, self.b]

    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(f"Dense attend une entrée (N, n_in), recu {x.shape} : oubli de Flatten ?")
        if self.W is None:
            self.n_in = x.shape[1]
            self.W = (torch.randn(self.n_in, self.n_out) / self.n_in ** 0.5).requires_grad_()
            self.b = torch.zeros(self.n_out, requires_grad=True)
        elif x.shape[1] != self.n_in:
            raise ValueError(f"Dense attend {self.n_in} entrées, recu {x.shape[1]}")
        return x @ self.W + self.b # logits (softmax dans la cross-entropy)

class Modele:
    def __init__(self, couches, input_shape=(1, 28, 28)):
        '''
        couches : liste de couches appliquées dans l'ordre
        input_shape : (C, H, W) d'une image, pour le passage à blanc qui fixe la taille des Dense
        '''

class CNN_torch:
    def fit(self, M):
        df = M.clone()
        if not self.fit_init:
            df, self.fit_init = self.fit_initialisation(df)

        N_out = torch.relu(self.convolution(df))  # Activation avec ReLu(x) = max(0,x) sur la sortie de convolution

        N_out = self.maxpooling(N_out)              # Maxpooling sur la sortie de ReLu.Convolution

        return N_out


Ton code est très bien, j'ai pu reprendre les différentes couches que j'avais deja faites (cellule 1 et 2 que tu m'a donné). Maintenant je passe au modèle et la suite, mais je voudrais le retrouver moi même pour reflechir et integrer, donne moi des premieres etapres larges, et je reviens vers toi si je galere (pour les étapes 3 à 7).

j'ai changé C en np.darray $\to$ tensor : Ca a tout cassé, faut reprendre toutes les fonctions tensor. (qui etait np. avant) pour reparer ca. Et tout les .shape ...

In [39]:
M = torch.tensor(np.array([[k for k in range(8)] for _ in range(8)]))
C = torch.tensor(np.array([[[k for _ in range(3)] for k in range(8)] for _ in range(8)]))
cnn = CNN_torch(C_out=6, kernel=3, stride=1, padding='valid')

# print(M)
# cnn.fit(M)

# print(C)
cnn.fit(C)

tensor([[[[ 0.0938, -0.0378,  1.4396],
          [-0.4997, -0.6821,  0.4134],
          [ 0.4069, -1.0837,  0.0048]],

         [[-1.6564, -1.6682,  1.0881],
          [ 0.8120,  0.2795,  0.8579],
          [ 0.1097,  0.5782, -0.4183]],

         [[-1.0810,  1.1069, -0.5872],
          [ 0.6069,  0.4366,  1.1926],
          [-0.6227, -0.2588,  0.8175]]],


        [[[ 0.5103,  1.3378,  2.0890],
          [ 1.1089,  0.4235,  0.2465],
          [-0.2875, -0.6971, -1.4186]],

         [[ 0.8143,  0.9633,  0.4557],
          [ 1.1613,  0.1663, -1.4542],
          [-0.3774,  0.4066,  0.2401]],

         [[ 0.6119,  0.3946, -0.3230],
          [-0.9497,  0.1472, -0.0502],
          [-1.1003, -0.2244,  0.0816]]],


        [[[-1.8823,  1.2626, -0.5354],
          [-1.8535, -0.1049, -0.4549],
          [-2.8137, -0.4256,  0.4642]],

         [[ 0.8318, -0.2046,  0.6509],
          [ 0.1910,  0.2005,  0.3004],
          [ 0.5426,  0.8006, -2.6952]],

         [[-0.7958,  1.7468,  0.4011],
     

tensor([[[9.9365e+00, 1.3234e+01, 1.6531e+01],
         [9.9365e+00, 1.3234e+01, 1.6531e+01],
         [9.9365e+00, 1.3234e+01, 1.6531e+01]],

        [[6.9283e+00, 1.5482e+01, 2.4035e+01],
         [6.9283e+00, 1.5482e+01, 2.4035e+01],
         [6.9283e+00, 1.5482e+01, 2.4035e+01]],

        [[2.4658e-02, 0.0000e+00, 0.0000e+00],
         [2.4658e-02, 0.0000e+00, 0.0000e+00],
         [2.4658e-02, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[1.1653e+01, 2.3529e+01, 3.5406e+01],
         [1.1653e+01, 2.3529e+01, 3.5406e+01],
         [1.1653e+01, 2.3529e+01, 3.5406e+01]]], grad_fn=<StackBackward0>)